# Final test-set evaluation on Colab (T21)

Scores the test split **once** with every final model: the n-grams at their best k, the interpolated
trigram, the eight context-ablation Transformers (T18) and the k=128 Transformers with a shorter
history (T19). Nothing gets tuned after this.

1. **Runtime → Change runtime type → T4 GPU**.
2. If the T21 code is not on `main` yet, set `BRANCH` in the clone cell below to your branch.
3. **Runtime → Run all**. It takes about 25 minutes; at the end a zip downloads.
4. If Colab disconnects, run all cells again.

In [ ]:
!nvidia-smi

Clone the repo, install it and build the token ids.

In [ ]:
BRANCH = "main"  # the branch that has the T21 code
REPO = "/content/language-models-from-scratch"
!test -d {REPO} || git clone --depth 1 --branch {BRANCH} https://github.com/mohamadalichamseddine/language-models-from-scratch.git {REPO}
%cd {REPO}
!pip install -q -r requirements.txt
!python scripts/prepare_data.py

Run the tests.

In [ ]:
!python -m pytest -q

## Score the test split

The 14 Transformer scorings run on the GPU first (about 15 minutes), then the n-grams on the CPU
(about 5 minutes), then the table with bootstrap confidence intervals. Models that already have
test predictions are skipped, so running this cell again only rebuilds the table.

In [ ]:
!python scripts/final_evaluation.py --device cuda

## Download the results

The zip holds `results/final/` (the table, its settings and the scoring log), the `metrics.json` of
every final run (now with test metrics next to the validation ones), and the `test.npz` predictions
that T22 and T23 need (about 40 MB).

In [ ]:
import subprocess
import zipfile
from pathlib import Path

from google.colab import files

modified = subprocess.run(["git", "ls-files", "--modified", "results"],
                          capture_output=True, text=True, check=True).stdout.split()
final = [p.as_posix() for p in Path("results/final").rglob("*") if p.is_file()]
predictions = [p.as_posix() for p in Path("results").rglob("test.npz")]
to_zip = sorted(set(modified + final + predictions))

with zipfile.ZipFile("/content/t21_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for name in to_zip:
        archive.write(name)
print(f"{len(to_zip)} files zipped:", *to_zip, sep="\n")
files.download("/content/t21_results.zip")